# XGBoost 二分类 — 完整流程 Demo

本 notebook 演示基于 `ml_tool` 工具包的 XGBoost 二分类完整流程，涵盖以下步骤：

1. 数据生成与数据集划分
2. 特征分析（缺失率、一值率、分位数、PSI、相关性）
3. WOE/IV 分箱
4. 特征筛选并导出报告
5. XGBoost 模型训练（Hyperopt 调参）
6. 自定义超参数搜索空间（可选）
7. 模型评估报告（分箱 Lift、按月 KS/AUC、特征重要性）
8. 特征分析报告
9. 模型保存与加载验证

In [ ]:
import os
import pandas as pd
import numpy as np
from ml_tool import FeatureAnalyzer, Binning, FeatureSelector, ModelTrainer, ReportGenerator, split_dataset

np.random.seed(42)
n = 3000
dates = pd.date_range("2024-01-01", periods=n, freq="D").to_series().sample(frac=1, random_state=42).values
df_raw = pd.DataFrame({
    "f1": np.random.randn(n),
    "f2": np.random.rand(n),
    "f3": np.random.randn(n) * 2,
    "f4": np.random.rand(n),
    "f5": np.random.randn(n),
    "y":  np.random.binomial(1, 0.15, n),
    "report_date": dates,
})
df_raw.loc[df_raw.sample(frac=0.05).index, "f2"] = np.nan
df = split_dataset(df_raw, date_col="report_date", oot_date="2024-09-01", train_ratio=0.8)
df["month"] = df["report_date"].dt.to_period("M").astype(str)

# 模拟附加列（在 split_dataset 调用之后添加到 df）
df["score_A"]    = np.random.uniform(300, 700, n)   # 模拟标品A分
df["score_B"]    = np.random.uniform(350, 750, n)   # 模拟标品B分
df["gain_score"] = np.random.uniform(0, 1, n)       # 模拟外部增益分
df["overdue_30"] = np.random.binomial(1, 0.12, n)   # 模拟逾期30天标签
df["overdue_60"] = np.random.binomial(1, 0.08, n)   # 模拟逾期60天标签

print(df.shape, df["dataset"].value_counts().sort_index().to_dict())

OUTPUT_DIR = "./output/xgb_clf"

## 1. 特征分析

In [ ]:
feature_cols = ["f1", "f2", "f3", "f4", "f5"]
analyzer = FeatureAnalyzer(df[feature_cols])

print("=== 缺失率 ==="); display(analyzer.missing_rate())
print("=== 一值率 ==="); display(analyzer.single_value_rate())
print("=== 分位数统计 ==="); display(analyzer.quantile_stats())

base_df    = df[df["dataset"] == "train"][feature_cols]
compare_df = df[df["dataset"] == "oot"][feature_cols]
psi_df = analyzer.psi(base_df, compare_df)
print("=== PSI（train vs OOT）==="); display(psi_df)
print("=== 相关性（|r|>0.3）==="); display(analyzer.correlation(threshold=0.3))

## 2. WOE/IV 分箱

In [ ]:
train_df = df[df["dataset"] == "train"].reset_index(drop=True)
binning = Binning(n_bins=10)
binning.fit(train_df[feature_cols + ["y"]], target="y", cols=feature_cols)
print("=== IV 汇总 ==="); display(binning.get_iv_summary())
print("=== f1 分箱详情 ==="); display(binning.get_iv_details("f1"))

## 3. 特征筛选

In [ ]:
selector = FeatureSelector(iv_threshold=0.001, psi_threshold=0.5, missing_threshold=0.98, corr_threshold=0.97)
selector.fit(df=train_df[feature_cols + ["y"]], target="y", base_df=base_df, compare_df=compare_df)
print("入选特征:", selector.get_selected())
print("剔除特征:", selector.dropped_cols)
display(selector.get_report())

reporter = ReportGenerator(output_dir=OUTPUT_DIR)
analysis = {
    "缺失率": analyzer.missing_rate(),
    "分位数统计": analyzer.quantile_stats(),
    "PSI": psi_df,
    "相关性": analyzer.correlation(threshold=0.0),
    "IV汇总": binning.get_iv_summary(),
}
path = reporter.feature_selection_report(selector.get_report(), analysis_results=analysis)
print("特征筛选报告:", path)

## 4. 模型训练（XGBoost 分类）

In [ ]:
selected = selector.get_selected()
X_train = df[df["dataset"] == "train"][selected]; y_train = df[df["dataset"] == "train"]["y"]
X_test  = df[df["dataset"] == "test"][selected];  y_test  = df[df["dataset"] == "test"]["y"]
X_oot   = df[df["dataset"] == "oot"][selected];   y_oot   = df[df["dataset"] == "oot"]["y"]

# 两轮流程：
#   第1轮 - 固定默认参数单次训练，用于特征筛选
#   第2轮 - 在筛选后特征上做 Hyperopt 超参搜索（n_trials 实际建议 100）
trainer = ModelTrainer(model_type="xgboost", n_trials=20)
trainer.fit(
    X_train, y_train, X_test, y_test, X_oot, y_oot,
    save_dir=OUTPUT_DIR,
)
print("入模特征数:", len(trainer.selected_features))
print("入模特征:", trainer.selected_features)
print("最优参数:", trainer.best_params)

In [ ]:
print("调参日志（按 loss 升序，前 10 行）:")
display(
    trainer.trials_log[["round", "trial", "loss", "is_best", "train_auc", "test_auc", "oot_auc"]]
    .sort_values("loss").head(10)
)

## 5. 自定义超参数空间（可选）

In [ ]:
from hyperopt import hp
import math

odds = float((y_train == 0).sum() / (y_train == 1).sum())

# ── 查看第一轮固定参数 ─────────────────────────────────────────
params1 = trainer.get_default_params(odds=odds)
print("第1轮固定参数:")
for k, v in params1.items():
    print("  {:30s}: {}".format(k, v))

# ── 查看第二轮默认搜索空间 ────────────────────────────────────
space = trainer.get_default_space(odds=odds)
print("\n第2轮搜索空间 key:")
for k, v in space.items():
    print("  {:30s}: {}".format(k, v))

In [ ]:
# 自定义第1轮固定参数
custom_p1 = {
    "max_depth": 3,
    "learning_rate": 0.03,
    "min_child_weight": max(int(odds * 5), 5),
    "gamma": 2,
}

# 自定义第2轮搜索空间
space["learning_rate"]    = hp.quniform("learning_rate", 0.01, 0.1, 0.01)
space["max_depth"]        = hp.choice("max_depth", [2, 3, 4])
space["min_child_weight"] = hp.quniform("min_child_weight", 5, 50, 5)
space["subsample"]        = hp.quniform("subsample", 0.6, 1.0, 0.1)

# 两轮流程：
#   第1轮 - 使用 custom_p1 固定参数单次训练
#   第2轮 - 使用自定义搜索空间做 Hyperopt 超参搜索
trainer_custom = ModelTrainer(model_type="xgboost", n_trials=10)
trainer_custom.fit(
    X_train, y_train, X_test, y_test, X_oot, y_oot,
    custom_params=custom_p1,
    custom_space=space,
)
print("最终 learning_rate:", trainer_custom.best_params.get("learning_rate"))
print("最终 max_depth:    ", trainer_custom.best_params.get("max_depth"))
print("最终 subsample:    ", trainer_custom.best_params.get("subsample"))

## 6. 模型评估报告

In [ ]:
import pandas as pd

# XGBoost 特征重要性从 booster.get_score 取
imp_dict = trainer._trainer.model.get_booster().get_score(importance_type="gain")
fi_df = pd.DataFrame(list(imp_dict.items()), columns=["feature", "importance"])

# 把预测概率列写回 df
df["y_pred_proba"] = float("nan")
df.loc[df["dataset"] == "train", "y_pred_proba"] = trainer.predict_proba(X_train)
df.loc[df["dataset"] == "test",  "y_pred_proba"] = trainer.predict_proba(X_test)
df.loc[df["dataset"] == "oot",   "y_pred_proba"] = trainer.predict_proba(X_oot)

clf_result = reporter.classification_report_from_df(
    df=df,
    target_col="y",
    pred_col="y_pred_proba",
    dataset_col="dataset",
    month_col="month",
    feature_importance=fi_df,
    gain_score_col="gain_score",
    gain_label_col="overdue_30",
    scorecard_cols=["score_A", "score_B"],
    scorecard_label_cols=["overdue_30", "overdue_60"],
    filename="XGBoost_分类评估报告",
)

display(clf_result["summary"])
print("Excel:", clf_result["excel_path"])
print("HTML: ", clf_result["html_path"])

In [ ]:
print('=== 分箱分析（训练集 / 验证集 / OOT 合并）===')
display(clf_result['sheets']['分箱分析'])

print('=== 按月KS_AUC ===')
if '按月KS_AUC' in clf_result['sheets']:
    display(clf_result['sheets']['按月KS_AUC'])

print('=== 按月分箱（三集合合并）===')
if '按月分箱' in clf_result['sheets']:
    display(clf_result['sheets']['按月分箱'])

## 7. 特征分析报告

In [ ]:
analysis_path = reporter.feature_analysis_report(
    df=df,
    features=feature_cols,
    target_col="y",
    dataset_col="dataset",
    month_col="month",
    filename="特征分析报告",
)
print("特征分析报告:", analysis_path)
import openpyxl
wb = openpyxl.load_workbook(analysis_path)
print("sheets:", wb.sheetnames)

## 8. 模型保存与加载

训练时传 `save_dir` 自动保存，各子目录结构如下：

- `01_feature_analysis/` — 特征分析中间结果
- `02_feature_selection/` — 特征筛选报告
- `03_tuning_log/` — 调参日志
- `04_evaluation/` — 模型评估报告（Excel + HTML）
- `05_model_deploy/model.pkl` — joblib 格式，含模型对象、入模特征、最优参数

支持两种加载方式：
1. `ModelTrainer.load()` — 通过 ml_tool 加载，保持完整接口
2. `joblib.load()` — 直接加载原生模型对象，适合生产打分

In [ ]:
import os, joblib

# ── 查看各子目录 ──────────────────────────────────────────────
print("OUTPUT_DIR 子目录:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    sub = os.path.join(OUTPUT_DIR, name)
    if os.path.isdir(sub):
        files = os.listdir(sub)
        print(f"  {name}/  -> {files}")
    else:
        print(f"  {name}")

# ── 方式一：ModelTrainer.load（保持完整接口）─────────────────
loaded = ModelTrainer.load(OUTPUT_DIR)
pred_via_trainer = loaded.predict_proba(X_oot)
print("\n[方式一] 入模特征:", loaded.selected_features)
print("[方式一] 预测前5个得分:", pred_via_trainer[:5].round(4))

# ── 方式二：joblib 直接加载原生模型，适合生产环境打分 ──────────
payload  = joblib.load(os.path.join(OUTPUT_DIR, "05_model_deploy", "model.pkl"))
model    = payload["model"]              # xgboost.XGBClassifier
features = payload["selected_features"]
best_p   = payload["best_params"]

print("\n[方式二] model_type:", payload["model_type"])
print("[方式二] 入模特征:", features)
print("[方式二] 最优参数（部分）:", {k: best_p[k] for k in list(best_p)[:3]})

# XGBClassifier 调用 predict_proba 取正类概率
pred_direct = model.predict_proba(X_oot[features])[:, 1]
print("[方式二] 预测前5个得分:", pred_direct[:5].round(4))

# 两种方式预测结果完全一致
max_diff = float(np.abs(pred_via_trainer - pred_direct).max())
print(f"\n两种方式最大偏差: {max_diff:.8f}（应为 0）")
assert max_diff == 0.0